In [ ]:
# only run this code for cluster 

import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, Subset, random_split, TensorDataset

import json 

import numpy as np

from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, roc_auc_score, precision_score, recall_score, f1_score, confusion_matrix
from sklearn.model_selection import train_test_split, StratifiedGroupKFold

import matplotlib.pyplot as plt
from sklearn.manifold import TSNE
from tqdm import tqdm

import sys
sys.path.append('../src')

from preprocessing import *
from models import  *
from utils import *

from autogluon.tabular import TabularPredictor

from tqdm import tqdm

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device

In [ ]:
dfs = get_dfs(os.path.dirname(os.getcwd()))
static_df = create_static_df(dfs)
medication_df = create_medication_df(dfs)
vitals_ca, vitals_lab = create_vitals_df(dfs)

ts_data = create_ts_data(vitals_ca, vitals_lab, medication_df, merge_lab=True, merge_med=True, static_df=static_df)

notes = create_notes_df(dfs, filename='../data/embeddings/emb_med_gte_simcse_en_ger.npy')

biopsy_df = dfs['biopsy']

all_valid_patient_ids = get_valid_patient_ids(
    static_df=static_df,
    ts_data=ts_data,
    notes_df=notes,
    min_ts_count=10,
    require_notes=True,
)

with open('../data/splits/pool_assignments.json') as f:
    pool_assignments = json.load(f)
pool_a_ids = np.asarray(pool_assignments['pool_a'])
pool_a_set = set(pool_a_ids.tolist())
selected_patient_ids = np.asarray([pid for pid in all_valid_patient_ids if pid in pool_a_set])

# Use the 90/10 backbone split to get preprocessing artifacts (matches training.ipynb)
backbone_split_path = '../data/splits/global_split_pool_a_9010.json'
backbone_split_ids = get_or_create_global_split(
    patient_ids=selected_patient_ids,
    split_json_path=backbone_split_path,
    train_size=0.9,
    val_size=0.1,
    test_size=0.0,
    random_state=42,
    shuffle=True,
    force_recreate=False,
)

# Fit preprocessing on backbone's train split (what the backbone saw during training)
preprocessing_ref = NephroCAGEDataset(
    static_df=static_df,
    ts_data=ts_data,
    notes_df=notes,
    biopsy_df=biopsy_df,
    patient_ids=backbone_split_ids['train'],
    fit_preprocessing=True,
    min_ts_count=10,
    require_notes=True,
)
preprocessing_artifacts = preprocessing_ref.preprocessing_artifacts

# Single dataset for ALL Pool A patients using backbone's preprocessing
all_dataset = NephroCAGEDataset(
    static_df=static_df,
    ts_data=ts_data,
    notes_df=notes,
    biopsy_df=biopsy_df,
    patient_ids=selected_patient_ids,
    preprocessing_artifacts=preprocessing_artifacts,
    fit_preprocessing=False,
    min_ts_count=10,
    require_notes=True,
)

print(f"Pool A size: {len(selected_patient_ids)}")
print(f"All-patient dataset size: {len(all_dataset)}")
print(f"Preprocessing fit on backbone train split ({len(backbone_split_ids['train'])} patients)")
print(f"Categorical cardinalities: {preprocessing_ref.categorical_cardinalities}")

In [ ]:
batch_size = 16
all_dataloader = DataLoader(all_dataset, batch_size=batch_size, shuffle=False, collate_fn=collate_fn)

att_encoder = TimeAwareAttentionEncoder(use_temporal_attention=True)
model = MultiModal(att_encoder, categorical_cardinalities=preprocessing_ref.categorical_cardinalities, use_static=True, use_notes=True).to(device)

model_path = '../models/backbone_poola_9010_best.pt'
checkpoint = torch.load(model_path, weights_only=False, map_location=device)
model.load_state_dict(checkpoint)
print(f"Loaded backbone from {model_path}")

In [ ]:
def extract_horizon_reprs(
    dataloader, 
    model, 
    horizons, 
    label_key, 
    rel_days_key=None,
    min_history_days=90,
    max_days=180,
    max_samples_per_patient=100,
    sampling_strategy="uniform",
    random_state=42
):
    """
    Max samples per patient is important. Used 100 here.
    The parameters min_history_days and max_days define the valid range of days for extracting representations.
    """
    from tqdm import tqdm
    import numpy as np
    import torch
    
    device = next(model.parameters()).device
    model.eval()

    hr_repr = {H: [] for H in horizons}
    hr_label = {H: [] for H in horizons}
    hr_days  = {H: [] for H in horizons}
    hr_pids  = {H: [] for H in horizons}

    all_pids = set()
    positive_pids = set()

    with torch.no_grad():
        for batch in tqdm(dataloader, desc="Extracting Timesteps"):
            pid  = batch['patient_id']
            slen = batch['seq_len']

            cat_static = batch['static_categorical_features'].to(device)
            num_static = batch['static_numerical_features'].to(device)

            full_ts    = batch['ts_features'].to(device)  # shape (B, T, F)
            timesteps  = batch['timesteps'].to(device)    # shape (B, T)
            mask_      = batch['mask'].to(device)         # shape (B, T)
            value_mask_full = batch['value_mask'].to(device)  # shape (B, T, F)

            raw_labels_data = batch[label_key]  # shape (B,)
            labels_data = []
            for val in raw_labels_data:
                if isinstance(val, torch.Tensor):
                    if val.numel() == 1:
                        val = float(val.item())
                    else:
                        val = val.cpu().numpy()
                labels_data.append(val)

            if rel_days_key and rel_days_key in batch:
                raw_rel_days_data = batch[rel_days_key]
                rel_days_data = []
                for dval in raw_rel_days_data:
                    if isinstance(dval, torch.Tensor):
                        if dval.numel() == 1:
                            dval = float(dval.item())
                        else:
                            dval = dval.cpu().numpy()
                    rel_days_data.append(dval)
                rel_days_data = np.array(rel_days_data)
            else:
                rel_days_data = None  

            notes_embeddings = batch['notes_embeddings'].to(device)
            notes_timesteps  = batch['notes_timesteps'].to(device)
            notes_mask       = batch['notes_mask'].to(device)

            B, T, F = full_ts.shape

            # Loop over each patient in this batch
            for i in range(B):
                patient_id_i = pid[i]
                all_pids.add(patient_id_i)

                label_or_list = labels_data[i]

                if isinstance(label_or_list, (int, float, np.number)):
                    if label_or_list == 1:
                        positive_pids.add(patient_id_i)
                elif isinstance(label_or_list, (list, np.ndarray)):
                    if len(label_or_list) > 0:
                        positive_pids.add(patient_id_i)
                elif label_or_list is not None:
                    raise TypeError(f"Unsupported label type: {type(label_or_list)}")

                if slen[i] < 2:
                    continue

                # Slice the valid portion of the time series: [0..slen[i]-1]
                seq_len_i = slen[i].item()
                ts_i = full_ts[i:i+1, :seq_len_i, :]   # (1, seq_len_i, F)
                tm_i = timesteps[i:i+1, :seq_len_i]    # (1, seq_len_i)
                mk_i = mask_[i:i+1, :seq_len_i]        # (1, seq_len_i)

                # Model input => omit last step from time series
                inp_seq = ts_i[:, :-1, :]              # (1, seq_len_i-1, F)
                inp_mask = mk_i[:, :-1]
                elapsed_times = build_elapsed_times(tm_i[:, :-1], inp_mask)
                inp_value_mask = value_mask_full[i:i+1, :seq_len_i-1, :]

                notes_emb_i = notes_embeddings[i:i+1]
                notes_ts_i  = notes_timesteps[i:i+1]
                notes_mk_i  = notes_mask[i:i+1]

                # Forward pass
                out, lstm_out, _, _ = model(
                    x=inp_seq,
                    elapsed_times=elapsed_times,     # (1, seq_len_i - 1)
                    timesteps=tm_i[:, :-1],          # (1, seq_len_i - 1)
                    notes_embeddings=notes_emb_i,
                    notes_timesteps=notes_ts_i,
                    static_features=(cat_static[i:i+1], num_static[i:i+1]),
                    mask=inp_mask,                   # (1, seq_len_i - 1)
                    notes_mask=notes_mk_i,
                    value_mask=inp_value_mask
                )
                
                # Time array for all steps
                time_arr = tm_i.cpu().numpy().flatten()  # (seq_len_i,)
                
                # Loop through all valid time steps
                for k in range(seq_len_i - 1):
                    cur_day = time_arr[k]  # Evaluate risk starting on the exact day of the measurement
                    
                    # Skip if outside the desired range
                    if cur_day < min_history_days or cur_day > max_days:
                        continue
                    
                    rep_ = lstm_out[0, k, :].cpu().numpy()
                    
                    # Process each horizon for this valid time step
                    for H in horizons:
                        
                        # If single-event logic is in play
                        if rel_days_data is not None:
                            event_label = label_or_list     # 0 or 1
                            event_day   = rel_days_data[i]  # single day
                            if (event_label == 1) and (0 < (event_day - cur_day) <= H):
                                label_ = 1
                            else:
                                label_ = 0

                        else:
                            if isinstance(label_or_list, (list, np.ndarray)) and len(label_or_list) > 0:
                                label_ = int(any(0 < (d - cur_day) <= H for d in label_or_list))
                            else:
                                label_ = 0

                        hr_repr[H].append(rep_)
                        hr_label[H].append(label_)
                        hr_days[H].append(cur_day)
                        hr_pids[H].append(patient_id_i)

    # Apply patient-level caps via sampling
    print("Applying patient-level caps...")
    if max_samples_per_patient is not None and max_samples_per_patient > 0:
        rng = np.random.default_rng(random_state)
        for H in horizons:
            if len(hr_pids[H]) == 0:
                continue

            pids_arr = np.array(hr_pids[H])
            keep_mask = np.zeros(len(pids_arr), dtype=bool)

            for pid_i in np.unique(pids_arr):
                idx = np.where(pids_arr == pid_i)[0]
                
                if len(idx) <= max_samples_per_patient:
                    chosen = list(idx)
                else:
                    if sampling_strategy == "first":
                        chosen = list(idx[:max_samples_per_patient])
                    elif sampling_strategy == "last":
                        chosen = list(idx[-max_samples_per_patient:])
                    elif sampling_strategy == "uniform":
                        pos = np.linspace(0, len(idx) - 1, num=max_samples_per_patient, dtype=int)
                        chosen = list(idx[pos])
                    elif sampling_strategy == "random":
                        chosen = list(rng.choice(idx, size=max_samples_per_patient, replace=False))
                    else:
                        raise ValueError(f"Unknown sampling_strategy: {sampling_strategy}")

                keep_mask[np.array(chosen, dtype=int)] = True

            # Apply the mask to filter the dataset
            kept_idx = np.where(keep_mask)[0]
            hr_repr[H] = [hr_repr[H][j] for j in kept_idx]
            hr_label[H] = [hr_label[H][j] for j in kept_idx]
            hr_days[H] = [hr_days[H][j] for j in kept_idx]
            hr_pids[H] = [hr_pids[H][j] for j in kept_idx]

    # Convert lists to numpy arrays for downstream compatibility
    for H in horizons:
        hr_repr[H] = np.array(hr_repr[H])
        hr_label[H] = np.array(hr_label[H])
        hr_days[H]  = np.array(hr_days[H])
        hr_pids[H]  = np.array(hr_pids[H])

    # Calculate and print final statistics
    for H in horizons:
        unique_pids, counts = np.unique(hr_pids[H], return_counts=True) if len(hr_pids[H]) else (np.array([]), np.array([]))
        total_patients = len(unique_pids)
        avg_samples = float(np.mean(counts)) if len(counts) else 0.0
        max_samples = int(np.max(counts)) if len(counts) else 0
        
        print(f"Horizon {H}: {total_patients} patients, avg {avg_samples:.1f} samples/patient, max {max_samples} samples/patient")

    print(f"Total unique patients processed: {len(all_pids)}, patients with at least one event: {len(positive_pids)}")
    return hr_repr, hr_label, hr_days, hr_pids

In [ ]:
horizons = [30, 90, 180]
min_history_days = 90
max_days = 720
max_samples_per_patient = 100
sampling_strategy = "uniform"

print("Extracting representations for all Pool A patients...\n")

all_graft_repr, all_graft_lbl, all_graft_days, all_graft_pids = extract_horizon_reprs(
    all_dataloader, model, horizons, "graft_loss_label", "loss_rel_days",
    min_history_days, max_days,
    max_samples_per_patient=max_samples_per_patient,
    sampling_strategy=sampling_strategy,
)

all_rej_repr, all_rej_lbl, all_rej_days, all_rej_pids = extract_horizon_reprs(
    all_dataloader, model, horizons, "rej_rel_days", None,
    min_history_days, max_days,
    max_samples_per_patient=max_samples_per_patient,
    sampling_strategy=sampling_strategy,
)

all_mort_repr, all_mort_lbl, all_mort_days, all_mort_pids = extract_horizon_reprs(
    all_dataloader, model, horizons, "death_label", "death_rel_days",
    min_history_days, max_days,
    max_samples_per_patient=max_samples_per_patient,
    sampling_strategy=sampling_strategy,
)

for H in horizons:
    print(f"\n===== Extracted Features for Horizon {H} days =====")
    print(f"Graft Loss: {all_graft_repr[H].shape}, pos rate: {all_graft_lbl[H].mean():.4f}")
    print(f"Rejection:  {all_rej_repr[H].shape}, pos rate: {all_rej_lbl[H].mean():.4f}")
    print(f"Mortality:  {all_mort_repr[H].shape}, pos rate: {all_mort_lbl[H].mean():.4f}")

In [ ]:
# Quick sanity check: logistic regression on 80/20 random split
def logistic_sanity_check(X, y, event_name="Event"):
    from sklearn.model_selection import train_test_split
    X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
    clf = LogisticRegression(class_weight={0:1, 1:10}, max_iter=1000).fit(X_tr, y_tr)
    y_proba = clf.predict_proba(X_te)[:, 1]
    auc = roc_auc_score(y_te, y_proba) if len(set(y_te)) > 1 else float('nan')
    print(f"  {event_name}: AUC={auc:.4f}")

print("Logistic Regression sanity check (quick 80/20):")
for H in horizons:
    logistic_sanity_check(all_graft_repr[H], all_graft_lbl[H], f"GraftLoss@{H}")
    logistic_sanity_check(all_rej_repr[H], all_rej_lbl[H], f"Rejection@{H}")
    logistic_sanity_check(all_mort_repr[H], all_mort_lbl[H], f"Mortality@{H}")

In [ ]:
def train_and_eval_mlp_fold(
    X_train, y_train,
    X_val, y_val,
    event_name="Event",
    epochs=30,
    batch_size=32,
    lr=5e-3,
    use_upsampling=True,
    eval_interval=1,
    min_checkpoint_epoch=1,
):
    """Train MLP for one CV fold; checkpoint on val AUC, return val metrics and best model state."""

    X_train_t = torch.tensor(X_train, dtype=torch.float32).to(device)
    y_train_t = torch.tensor(y_train, dtype=torch.float32).to(device)
    X_val_t = torch.tensor(X_val, dtype=torch.float32).to(device)

    train_ds = TensorDataset(X_train_t, y_train_t)
    train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True)

    model_mlp = SimpleMLP(input_dim=X_train.shape[1]).to(device)
    optimizer = optim.Adam(model_mlp.parameters(), lr=lr, weight_decay=1e-4)

    if use_upsampling:
        num_pos = max(1, np.sum(y_train == 1))
        num_neg = np.sum(y_train == 0)
        dynamic_weight = num_neg / num_pos
        pos_weight = torch.tensor([dynamic_weight], device=device)
        criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
    else:
        criterion = nn.BCEWithLogitsLoss()

    def predict_proba(model_eval, X_t):
        model_eval.eval()
        with torch.no_grad():
            logits = model_eval(X_t)
            return torch.sigmoid(logits).cpu().numpy().reshape(-1)

    def select_threshold(y_true, probs):
        y_true = np.asarray(y_true).astype(int)
        candidates = np.unique(np.concatenate([np.linspace(0.05, 0.95, 19), probs]))
        best_thr, best_f1 = 0.5, -1.0
        for thr in candidates:
            f1 = f1_score(y_true, (probs >= thr).astype(int), zero_division=0)
            if f1 > best_f1 or (np.isclose(f1, best_f1) and abs(thr - 0.5) < abs(best_thr - 0.5)):
                best_f1, best_thr = f1, float(thr)
        return best_thr

    def evaluate_split(y_true, probs, threshold):
        true_np = np.asarray(y_true).astype(int)
        pred = (probs >= threshold).astype(int)
        auc = roc_auc_score(true_np, probs) if len(set(true_np)) > 1 else float('nan')
        tn, fp, fn, tp = confusion_matrix(true_np, pred).ravel()
        return {
            'auc': auc,
            'acc': accuracy_score(true_np, pred),
            'prec': precision_score(true_np, pred, zero_division=0),
            'recall': recall_score(true_np, pred, zero_division=0),
            'spec': tn / (tn + fp) if (tn + fp) > 0 else 0.0,
            'f1': f1_score(true_np, pred, zero_division=0),
        }

    best_val_auc = -1.0
    best_state = None
    best_epoch = 0
    best_threshold = 0.5

    for epoch in range(1, epochs + 1):
        model_mlp.train()
        for batch_x, batch_y in train_loader:
            optimizer.zero_grad()
            loss = criterion(model_mlp(batch_x), batch_y)
            loss.backward()
            optimizer.step()

        should_eval = (epoch % eval_interval == 0) or (epoch == epochs)
        if should_eval:
            val_probs = predict_proba(model_mlp, X_val_t)
            threshold = select_threshold(y_val, val_probs)
            val_auc = roc_auc_score(y_val.astype(int), val_probs) if len(set(y_val.astype(int))) > 1 else float('nan')

            if epoch >= min_checkpoint_epoch and not np.isnan(val_auc) and val_auc > best_val_auc:
                best_val_auc = val_auc
                best_state = {k: v.detach().cpu().clone() for k, v in model_mlp.state_dict().items()}
                best_epoch = epoch
                best_threshold = threshold

    if best_state is None:
        best_state = {k: v.detach().cpu().clone() for k, v in model_mlp.state_dict().items()}
        best_epoch = epochs
        best_threshold = select_threshold(y_val, predict_proba(model_mlp, X_val_t))

    model_mlp.load_state_dict(best_state)
    val_probs = predict_proba(model_mlp, X_val_t)
    metrics = evaluate_split(y_val, val_probs, best_threshold)
    metrics['epoch'] = best_epoch
    metrics['threshold'] = best_threshold
    return metrics, best_state

In [ ]:
n_splits = 5
skf = StratifiedGroupKFold(n_splits=n_splits, shuffle=True, random_state=42)

tasks = {
    'GraftLoss': (all_graft_repr, all_graft_lbl, all_graft_days, all_graft_pids),
    'Rejection': (all_rej_repr, all_rej_lbl, all_rej_days, all_rej_pids),
    'Mortality': (all_mort_repr, all_mort_lbl, all_mort_days, all_mort_pids),
}

cv_results = {}
saved_models = {}

for task_name, (repr_dict, lbl_dict, days_dict, pids_dict) in tasks.items():
    for H in horizons:
        key = f"{task_name}@{H}"
        save_path = f'../models/{key}_clf.pth'

        if os.path.exists(save_path):
            checkpoint = torch.load(save_path, weights_only=False, map_location=device)
            if 'cv_metrics' in checkpoint:
                cv_results[key] = checkpoint['cv_metrics']
                print(f"\n{'='*60}")
                print(f"SKIP {key}: loaded from {save_path} (CV AUC={cv_results[key]['auc']:.4f})")
                if 'fold_metrics' in checkpoint:
                    for i, fm in enumerate(checkpoint['fold_metrics']):
                        print(f"  Fold {i}: AUC={fm['auc']:.4f}, F1={fm['f1']:.4f}, Ep={fm['epoch']}, Thr={fm['threshold']:.3f}")
            else:
                print(f"\n{'='*60}")
                print(f"SKIP {key}: {save_path} exists (no saved metrics, re-delete to retrain)")
            saved_models[key] = save_path
            continue

        X_all = repr_dict[H]
        y_all = lbl_dict[H]
        pids_all = pids_dict[H]

        unique_pids_arr = np.unique(pids_all)
        pid_to_label = {}
        for pid in unique_pids_arr:
            pid_to_label[pid] = int(y_all[pids_all == pid].any())

        strat_y = np.array([pid_to_label[p] for p in pids_all])
        n_pos_patients = sum(pid_to_label.values())
        n_neg_patients = len(pid_to_label) - n_pos_patients

        print(f"\n{'='*60}")
        print(f"{key}: {len(unique_pids_arr)} patients ({n_pos_patients} pos, {n_neg_patients} neg), {len(y_all)} timesteps")
        print(f"{'='*60}")

        fold_metrics_list = []
        best_fold_auc = -1.0
        best_fold_state = None
        best_fold_idx = 0

        for fold, (train_idx, val_idx) in enumerate(skf.split(X_all, strat_y, groups=pids_all)):
            X_train, y_train = X_all[train_idx], y_all[train_idx]
            X_val, y_val = X_all[val_idx], y_all[val_idx]

            n_train_pos = int(np.sum(y_train == 1))
            n_val_pos = int(np.sum(y_val == 1))

            metrics, fold_state = train_and_eval_mlp_fold(
                X_train, y_train, X_val, y_val,
                event_name=f"{key}_fold{fold}",
                epochs=15, batch_size=32, lr=5e-3,
                use_upsampling=True, eval_interval=1, min_checkpoint_epoch=1,
            )
            fold_metrics_list.append(metrics)

            if not np.isnan(metrics['auc']) and metrics['auc'] > best_fold_auc:
                best_fold_auc = metrics['auc']
                best_fold_state = fold_state
                best_fold_idx = fold

            print(
                f"  Fold {fold}: AUC={metrics['auc']:.4f}, F1={metrics['f1']:.4f}, "
                f"Ep={metrics['epoch']}, Thr={metrics['threshold']:.3f} "
                f"(train: {len(y_train)} [{n_train_pos} pos], val: {len(y_val)} [{n_val_pos} pos])"
            )

        # Aggregate fold metrics
        avg_metrics = {}
        for mk in ['auc', 'acc', 'prec', 'recall', 'spec', 'f1']:
            vals = [fm[mk] for fm in fold_metrics_list if not np.isnan(fm[mk])]
            avg_metrics[mk] = np.mean(vals) if vals else float('nan')
            avg_metrics[f"{mk}_std"] = np.std(vals) if vals else float('nan')

        cv_results[key] = avg_metrics

        # Save model weights + CV metrics + per-fold metrics
        torch.save({
            'model_state_dict': best_fold_state,
            'best_fold': best_fold_idx,
            'cv_metrics': avg_metrics,
            'fold_metrics': fold_metrics_list,
        }, save_path)
        saved_models[key] = save_path
        print(f"  Saved classifier head (fold {best_fold_idx}) to {save_path} (val AUC={best_fold_auc:.4f})")
        print(f"  >>> {key} CV AUC: {avg_metrics['auc']:.4f} +/- {avg_metrics['auc_std']:.4f}")

print(f"\n{'='*60}")
print("Saved classifier heads:")
for key, path in saved_models.items():
    print(f"  {key}: {path}")

In [ ]:
print(f"\n{'='*70}")
print("SUMMARY — 5-Fold StratifiedGroupKFold CV (mean +/- std AUC)")
print(f"{'='*70}")
print(f"{'Event':<20} {'30-day':>18} {'90-day':>18} {'180-day':>18}")
for event in ['GraftLoss', 'Rejection', 'Mortality']:
    vals = []
    for H in [30, 90, 180]:
        m = cv_results[f"{event}@{H}"]
        vals.append(f"{m['auc']:.4f}+/-{m['auc_std']:.4f}")
    print(f"{event:<20} {vals[0]:>18} {vals[1]:>18} {vals[2]:>18}")

print(f"\n{'='*70}")
print("Detailed CV Metrics (mean)")
print(f"{'='*70}")
print(f"{'Event':<20} {'AUC':>15} {'Acc':>8} {'Prec':>8} {'Recall':>8} {'Spec':>8} {'F1':>8}")
for k, v in cv_results.items():
    print(
        f"{k:<20} {v['auc']:.4f}+/-{v['auc_std']:.4f}"
        f" {v['acc']:.4f} {v['prec']:.4f} {v['recall']:.4f} {v['spec']:.4f} {v['f1']:.4f}"
    )

print(f"\nAll CV results: {json.dumps({k: {mk: round(mv, 4) for mk, mv in v.items()} for k, v in cv_results.items()}, indent=2)}")